# 🔬 Semantic Segmentation with PyTorch

В этом воркшопе разберём несколько подходов к **семантической сегментации**:

| # | Раздел | Что изучаем |
|:-:|:------:|:------------|
| 1 | Dataset & Augmentations | Pets dataset, Albumentations |
| 2 | UNet (улучшенный) | Полноценная архитектура с Bottleneck и Dropout |
| 3 | Метрики | Dice, IoU, визуализация |
| 4 | SMP библиотека | Unet + FPN + DeepLabV3+ с pretrained backbone |
| 5 | Сравнение моделей | Задания |

![](https://lmb.informatik.uni-freiburg.de/people/ronneber/u-net/u-net-architecture.png)


## 📦 Импорты и настройка

In [1]:
import random, os, copy, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.transforms as T
from torchvision.datasets.utils import download_and_extract_archive

from tqdm.notebook import tqdm, trange
from PIL import Image

try:
    from torchsummary import summary
except:
    %pip install torchsummary -q
    from torchsummary import summary

try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
except:
    %pip install albumentations -q
    import albumentations as A
    from albumentations.pytorch import ToTensorV2

print("✅ Импорты выполнены")


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Ronkin\anaconda3\envs\ts\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Ronkin\anaconda3\envs\ts\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\Ronkin\anaconda3\envs\ts\Lib\site-packages\ipykernel\kernelapp.py", line 758, in start
    self.io_loop.start()
  File "C:\Users\Ro

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

In [ ]:
def torch_settings():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    dtype  = torch.float32
    print(f'Device: {device} | PyTorch: {torch.__version__}')
    if device.type == 'cuda':
        print(f'  GPU: {torch.cuda.get_device_name(0)}')
    torch.set_default_dtype(dtype)
    return device, dtype, os.cpu_count()

def torch_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

device, dtype, num_workers = torch_settings()
torch_seed()


---
## 🐾 Часть 1. Dataset — Oxford-IIIT Pets

Датасет содержит фотографии кошек и собак с масками сегментации (3 класса):
- `1` — силуэт животного → мы переводим в `1`
- `2` — фон → мы переводим в `0`  
- `3` — граница маски → мы переводим в `1`


In [ ]:
# Скачиваем датасет
root_directory = os.path.join(os.getcwd(), 'data', 'pets')

for url in ["https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz",
            "https://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz"]:
    try:
        download_and_extract_archive(url, root_directory)
    except:
        print(f"Уже скачан или ошибка: {url.split('/')[-1]}")

images_directory = os.path.join(root_directory, "images")
masks_directory  = os.path.join(root_directory, "annotations", "trimaps")
print(f"Изображений: {len(os.listdir(images_directory))}")


In [2]:
N_TRAIN, N_TEST, STEP = 6000, 100, 5

images_filenames = [f for f in os.listdir(images_directory)
                    if os.path.splitext(f)[-1] in ('.png', '.jpg', '.jpeg')]
random.shuffle(images_filenames)

train_filenames = images_filenames[:N_TRAIN:STEP]
val_filenames   = images_filenames[N_TRAIN:-N_TEST:STEP]
test_filenames  = images_filenames[-N_TEST::STEP]

print(f"Train: {len(train_filenames)} | Val: {len(val_filenames)} | Test: {len(test_filenames)}")


NameError: name 'images_directory' is not defined

In [3]:
class PetsDataset(Dataset):
    def __init__(self, filenames, images_dir, masks_dir, transforms=None):
        self.filenames  = filenames
        self.images_dir = images_dir
        self.masks_dir  = masks_dir
        self.transforms = transforms

    def __len__(self): return len(self.filenames)

    def preprocess_mask(self, mask):
        mask = np.array(mask).astype(np.float32)
        mask[mask == 2.0] = 0.0
        mask[(mask == 1.0) | (mask == 3.0)] = 1.0
        return mask

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        image = np.array(Image.open(os.path.join(self.images_dir, fname)).convert("RGB"))
        mask  = self.preprocess_mask(
                    Image.open(os.path.join(self.masks_dir, fname.replace(".jpg", ".png"))))

        if self.transforms:
            t = self.transforms(image=image, mask=mask)
            image, mask = t["image"], t["mask"]
        return image, mask


In [ ]:
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

train_transform = A.Compose([
    A.Resize(280, 280),
    A.RandomCrop(256, 256),
    A.HorizontalFlip(p=0.5),
    A.Affine(scale=[0.85, 1.15], rotate=[-15, 15], p=0.5),
    A.RGBShift(25, 25, 25, p=0.4),
    A.RandomBrightnessContrast(0.3, 0.3, p=0.4),
    A.GaussNoise(p=0.2),
    A.Normalize(mean, std),
    ToTensorV2(),
])

test_transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(mean, std),
    ToTensorV2(),
])

BATCH_SIZE = 8

train_dataset = PetsDataset(train_filenames, images_directory, masks_directory, train_transform)
val_dataset   = PetsDataset(val_filenames,   images_directory, masks_directory, test_transform)
test_dataset  = PetsDataset(test_filenames,  images_directory, masks_directory, test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")


In [ ]:
def denormalize(tensor, mean=mean, std=std):
    """Отменяет нормализацию для визуализации."""
    t = tensor.clone().permute(1, 2, 0).numpy()
    t = t * np.array(std) + np.array(mean)
    return np.clip(t, 0, 1)

def visualize_samples(dataset, n=4, title="Примеры датасета"):
    fig, axes = plt.subplots(2, n, figsize=(3*n, 6))
    for i in range(n):
        img, mask = dataset[i]
        axes[0, i].imshow(denormalize(img))
        axes[0, i].set_title(f"Изображение {i}")
        axes[0, i].axis('off')
        axes[1, i].imshow(mask.numpy(), cmap='gray')
        axes[1, i].set_title(f"Маска {i}")
        axes[1, i].axis('off')
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

visualize_samples(val_dataset, n=4)


---
## 🏗️ Часть 2. UNet — улучшенная архитектура

По сравнению с упрощённой версией добавлено:
- **Bottleneck** блок между энкодером и декодером
- **Dropout** в энкодере для регуляризации
- **ConvTranspose2d** вместо Upsample (опционально)
- Корректная инициализация весов (He initialization)

### Архитектура:
```
Input (3, 256, 256)
  ↓ Encoder
  E1: 64 ch  ─────────────────────── skip1
  E2: 128 ch ──────────────── skip2  │
  E3: 256 ch ────────── skip3  │     │
  E4: 512 ch ──── skip4  │     │     │
  Bottleneck: 1024 ch    │     │     │
  ↓ Decoder              │     │     │
  D4: 512 ch ←──── cat ──┘     │     │
  D3: 256 ch ←──── cat ────────┘     │
  D2: 128 ch ←──── cat ──────────────┘
  D1: 64 ch  ←──── cat ──────────────────── skip1
  Output (1, 256, 256)
```


In [ ]:
# ── Строительные блоки ────────────────────────────────────────────────────────

class DoubleConv(nn.Module):
    """Два свёрточных слоя: Conv → BN → ReLU → Conv → BN → ReLU"""
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
        if dropout > 0:
            layers.append(nn.Dropout2d(dropout))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class EncoderBlock(nn.Module):
    """MaxPool → DoubleConv"""
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.conv = DoubleConv(in_ch, out_ch, dropout)

    def forward(self, x):
        return self.conv(self.pool(x))


class DecoderBlock(nn.Module):
    """Upsample / ConvTranspose → concat skip → DoubleConv"""
    def __init__(self, in_ch, skip_ch, out_ch, use_transpose=False):
        super().__init__()
        if use_transpose:
            self.up = nn.ConvTranspose2d(in_ch, in_ch // 2, kernel_size=2, stride=2)
            conv_in = in_ch // 2 + skip_ch
        else:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            conv_in = in_ch + skip_ch
        self.conv = DoubleConv(conv_in, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        # Выравниваем размеры на случай нечётных размеров
        if x.shape != skip.shape:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)


In [ ]:
class UNet(nn.Module):
    """
    Полноценный UNet с:
    - Энкодер: 4 уровня (64 → 128 → 256 → 512)
    - Bottleneck: 1024 каналов
    - Декодер: 4 уровня с skip-connections
    - Dropout для регуляризации
    - Опция: ConvTranspose вместо Bilinear upsample
    """
    def __init__(self, n_channels=3, n_classes=1,
                 base_ch=64, dropout=0.1, use_transpose=False):
        super().__init__()

        # Энкодер
        self.enc1 = DoubleConv(n_channels, base_ch)           # 256 → 256
        self.enc2 = EncoderBlock(base_ch,     base_ch*2,  dropout)  # 256 → 128
        self.enc3 = EncoderBlock(base_ch*2,   base_ch*4,  dropout)  # 128 → 64
        self.enc4 = EncoderBlock(base_ch*4,   base_ch*8,  dropout)  #  64 → 32

        # Bottleneck
        self.bottleneck = EncoderBlock(base_ch*8, base_ch*16, dropout)  # 32 → 16

        # Декодер
        self.dec4 = DecoderBlock(base_ch*16, base_ch*8,  base_ch*8,  use_transpose)
        self.dec3 = DecoderBlock(base_ch*8,  base_ch*4,  base_ch*4,  use_transpose)
        self.dec2 = DecoderBlock(base_ch*4,  base_ch*2,  base_ch*2,  use_transpose)
        self.dec1 = DecoderBlock(base_ch*2,  base_ch,    base_ch,    use_transpose)

        # Выходной слой
        self.out_conv = nn.Conv2d(base_ch, n_classes, kernel_size=1)

        # Инициализация весов
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        # Энкодер (сохраняем skip-connections)
        s1 = self.enc1(x)          # [B, 64,  H,   W  ]
        s2 = self.enc2(s1)         # [B, 128, H/2, W/2]
        s3 = self.enc3(s2)         # [B, 256, H/4, W/4]
        s4 = self.enc4(s3)         # [B, 512, H/8, W/8]

        # Bottleneck
        b  = self.bottleneck(s4)   # [B, 1024, H/16, W/16]

        # Декодер
        x = self.dec4(b,  s4)      # [B, 512, H/8, W/8]
        x = self.dec3(x,  s3)      # [B, 256, H/4, W/4]
        x = self.dec2(x,  s2)      # [B, 128, H/2, W/2]
        x = self.dec1(x,  s1)      # [B, 64,  H,   W  ]

        return self.out_conv(x)    # [B, n_classes, H, W]


# Создаём модель и выводим summary
model = UNet(n_channels=3, n_classes=1, base_ch=32, dropout=0.1)
summary(model, input_size=(3, 256, 256), device='cpu')


---
## 📐 Часть 3. Функции потерь и метрики

### Dice Loss
$$\text{Dice} = 1 - \frac{2 \cdot |A \cap B| + \epsilon}{|A| + |B| + \epsilon}$$

### IoU (Intersection over Union / Jaccard Index)
$$\text{IoU} = \frac{|A \cap B|}{|A \cup B|} = \frac{|A \cap B|}{|A| + |B| - |A \cap B|}$$

![](https://miro.medium.com/max/858/1*yUd5ckecHjWZf6hGrdlwzA.png)


In [ ]:
# ── Функции потерь ─────────────────────────────────────────────────────────────

def dice_coeff(pred, target, smooth=1.0):
    """Dice coefficient (1 = полное совпадение)."""
    pred   = pred.view(-1)
    target = target.view(-1)
    inter  = (pred * target).sum()
    return (2.0 * inter + smooth) / (pred.sum() + target.sum() + smooth)


def iou_score(pred, target, threshold=0.5, smooth=1.0):
    """IoU / Jaccard index (бинаризованный)."""
    pred_bin = (pred > threshold).float()
    pred_b   = pred_bin.view(-1)
    target_b = target.view(-1)
    inter    = (pred_b * target_b).sum()
    union    = pred_b.sum() + target_b.sum() - inter
    return (inter + smooth) / (union + smooth)


class DiceLoss(nn.Module):
    def forward(self, logits, target, smooth=1.0):
        pred = torch.sigmoid(logits)
        return 1.0 - dice_coeff(pred, target, smooth)


class DiceBCELoss(nn.Module):
    """BCE + Dice Loss. weight контролирует баланс."""
    def __init__(self, weight=0.5):
        super().__init__()
        self.w = weight
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, target):
        bce  = self.bce(logits, target)
        pred = torch.sigmoid(logits)
        dice = 1.0 - dice_coeff(pred, target)
        return self.w * bce + (1 - self.w) * dice


def accuracy(y_pred, y):
    """Dice accuracy для мониторинга обучения."""
    pred = torch.sigmoid(y_pred)
    return dice_coeff(pred, y)


# Быстрая проверка
dummy_pred = torch.randn(4, 1, 256, 256)
dummy_mask = torch.randint(0, 2, (4, 256, 256)).float()

criterion = DiceBCELoss(weight=0.5)
loss_val = criterion(dummy_pred, dummy_mask)
print(f"Loss (random pred): {loss_val.item():.4f}  ← должно быть ~0.7-0.9")


## 🏋️ Часть 4. Обучение

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = total_acc = 0
    for x, y in tqdm(loader, desc="Train", leave=False):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        total_acc  += accuracy(pred, y).item()
    n = len(loader)
    return total_loss / n, total_acc / n


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = total_acc = total_iou = 0
    for x, y in tqdm(loader, desc="Eval", leave=False):
        x, y = x.to(device), y.to(device)
        pred = model(x)
        total_loss += criterion(pred, y).item()
        total_acc  += accuracy(pred, y).item()
        total_iou  += iou_score(torch.sigmoid(pred), y).item()
    n = len(loader)
    return total_loss / n, total_acc / n, total_iou / n


def fit(model, train_loader, val_loader, optimizer, criterion,
        epochs=10, device='cpu', scheduler=None, save_path='best_model.pt'):

    best_loss = float('inf')
    history   = {k: [] for k in ['train_loss','val_loss','train_dice','val_dice','val_iou']}

    for epoch in trange(epochs, desc="Epochs"):
        t0 = time.monotonic()
        tr_loss, tr_dice = train_epoch(model, train_loader, optimizer, criterion, device)
        vl_loss, vl_dice, vl_iou = eval_epoch(model, val_loader, criterion, device)

        if scheduler: scheduler.step()

        if vl_loss < best_loss:
            best_loss = vl_loss
            torch.save(model.state_dict(), save_path)

        for k, v in zip(['train_loss','val_loss','train_dice','val_dice','val_iou'],
                         [tr_loss, vl_loss, tr_dice, vl_dice, vl_iou]):
            history[k].append(v)

        mins = int((time.monotonic()-t0)//60)
        secs = int((time.monotonic()-t0) % 60)
        if epoch % 2 == 1:
            print(f"Ep {epoch+1:02d}/{epochs} [{mins}m{secs}s] "
                  f"Loss: {tr_loss:.3f}/{vl_loss:.3f} | "
                  f"Dice: {tr_dice*100:.1f}%/{vl_dice*100:.1f}% | "
                  f"IoU: {vl_iou*100:.1f}%")
    return history


In [ ]:
# ── Конфигурация обучения ─────────────────────────────────────────────────────
LR     = 1e-3
EPOCHS = 10

model     = UNet(n_channels=3, n_classes=1, base_ch=32, dropout=0.1).to(device)
criterion = DiceBCELoss(weight=0.5).to(device)
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print(f"Параметров: {sum(p.numel() for p in model.parameters()):,}")


In [ ]:
history = fit(model, train_loader, val_loader,
             optimizer, criterion,
             epochs=EPOCHS, device=device,
             scheduler=scheduler, save_path='unet_best.pt')


In [ ]:
# ── Графики обучения ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'],   label='Val')
axes[0].set_title('Loss (DiceBCE)')
axes[0].legend(); axes[0].grid(True)

axes[1].plot([d*100 for d in history['train_dice']], label='Train')
axes[1].plot([d*100 for d in history['val_dice']],   label='Val')
axes[1].set_title('Dice Score (%)')
axes[1].legend(); axes[1].grid(True)

axes[2].plot([v*100 for v in history['val_iou']])
axes[2].set_title('Validation IoU (%)')
axes[2].grid(True)

plt.tight_layout()
plt.show()


---
## 🖼️ Визуализация предсказаний

Смотрим на качество сегментации: исходное изображение, ground truth маска и предсказание.


In [ ]:
THRESHOLD = 0.5   # порог бинаризации

def visualize_predictions(model, dataset, n=4, threshold=THRESHOLD, device=device):
    model.eval()
    fig, axes = plt.subplots(n, 3, figsize=(10, 3*n))

    for i in range(n):
        img, mask_gt = dataset[i]
        with torch.no_grad():
            pred = torch.sigmoid(model(img.unsqueeze(0).to(device)))
            pred = pred.squeeze().cpu().numpy()

        mask_pred_bin = pred >= threshold

        # Dice и IoU для этого примера
        dice = 2*(mask_pred_bin * mask_gt.numpy()).sum() /                (mask_pred_bin.sum() + mask_gt.numpy().sum() + 1e-6)
        iou  = (mask_pred_bin * mask_gt.numpy()).sum() /                ((mask_pred_bin | mask_gt.numpy().astype(bool)).sum() + 1e-6)

        axes[i, 0].imshow(denormalize(img))
        axes[i, 0].set_title('Изображение'); axes[i, 0].axis('off')

        axes[i, 1].imshow(mask_gt.numpy(), cmap='gray')
        axes[i, 1].set_title('Ground Truth'); axes[i, 1].axis('off')

        axes[i, 2].imshow(mask_pred_bin, cmap='gray')
        axes[i, 2].set_title(f'Pred  Dice={dice:.2f}  IoU={iou:.2f}')
        axes[i, 2].axis('off')

    plt.suptitle(f'UNet Predictions (threshold={threshold})', fontsize=14)
    plt.tight_layout()
    plt.show()

# Загружаем лучшие веса
model.load_state_dict(torch.load('unet_best.pt'))
visualize_predictions(model, val_dataset, n=4)


In [ ]:
# ── Влияние порога бинаризации ────────────────────────────────────────────────
def threshold_sweep(model, dataset, n_samples=50, device=device):
    model.eval()
    thresholds = np.arange(0.1, 0.95, 0.05)
    dice_scores = []

    for thr in thresholds:
        dice_sum = 0
        for i in range(n_samples):
            img, mask_gt = dataset[i]
            with torch.no_grad():
                pred = torch.sigmoid(model(img.unsqueeze(0).to(device)))
                pred = pred.squeeze().cpu().numpy()
            pred_bin = pred >= thr
            dice_sum += 2*(pred_bin * mask_gt.numpy()).sum() /                         (pred_bin.sum() + mask_gt.numpy().sum() + 1e-6)
        dice_scores.append(dice_sum / n_samples)

    best_thr = thresholds[np.argmax(dice_scores)]
    plt.figure(figsize=(7, 4))
    plt.plot(thresholds, dice_scores, 'o-', color='steelblue')
    plt.axvline(best_thr, color='red', linestyle='--', label=f'Best={best_thr:.2f}')
    plt.xlabel('Threshold'); plt.ylabel('Dice Score')
    plt.title('Dice vs Threshold'); plt.legend(); plt.grid(True)
    plt.show()
    print(f"Лучший порог: {best_thr:.2f}  →  Dice: {max(dice_scores):.4f}")
    return best_thr

best_threshold = threshold_sweep(model, val_dataset)


---
## 🚀 Часть 5. `segmentation_models_pytorch` (SMP)

Библиотека SMP предоставляет готовые архитектуры с **pretrained энкодерами**.

### Доступные архитектуры:

| Архитектура | Идея | Когда использовать |
|:-----------:|:----:|:-----------------:|
| **UNet** | Skip-connections (попиксельное) | Медицина, тонкие объекты |
| **FPN** | Feature Pyramid Network (мультимасштаб) | Объекты разных размеров |
| **DeepLabV3+** | Dilated conv + ASPP | Точные границы |
| **Linknet** | Лёгкий энкодер-декодер | Мобильные приложения |

### Доступные энкодеры (backbone):
`resnet34`, `efficientnet-b0`, `mobilenet_v2`, `mit_b0`, ...


In [ ]:
try:
    import segmentation_models_pytorch as smp
except ImportError:
    %pip install segmentation_models_pytorch -q
    import segmentation_models_pytorch as smp

print("SMP version:", smp.__version__)


In [ ]:
# ── UNet с pretrained ResNet34 энкодером ─────────────────────────────────────

model_smp = smp.Unet(
    encoder_name    = "resnet34",      # backbone
    encoder_weights = "imagenet",      # pretrained веса
    in_channels     = 3,
    classes         = 1,
    activation      = None,            # sigmoid применим сами
)

# Считаем параметры по группам
enc_params = sum(p.numel() for p in model_smp.encoder.parameters())
dec_params = sum(p.numel() for p in model_smp.decoder.parameters())
total      = sum(p.numel() for p in model_smp.parameters())

print(f"Энкодер (ResNet34): {enc_params:>10,} параметров")
print(f"Декодер:            {dec_params:>10,} параметров")
print(f"Итого:              {total:>10,} параметров")
print()
summary(model_smp, input_size=(3, 256, 256), device='cpu')


### Transfer Learning стратегия

При работе с pretrained моделями есть два подхода:

1. **Fine-tuning всей сети** — обновляем все веса (нужно много данных)
2. **Freeze encoder** — сначала обучаем только декодер, потом размораживаем

Второй подход особенно полезен при маленьких датасетах.


In [ ]:
def freeze_encoder(model):
    """Замораживает веса энкодера."""
    for param in model.encoder.parameters():
        param.requires_grad = False
    n_frozen = sum(p.numel() for p in model.encoder.parameters())
    print(f"Заморожено {n_frozen:,} параметров энкодера")


def unfreeze_encoder(model):
    """Размораживает веса энкодера."""
    for param in model.encoder.parameters():
        param.requires_grad = True
    print("Энкодер разморожен")


# Стратегия: сначала быстро учим декодер, потом fine-tune всё
model_smp = model_smp.to(device)

# Этап 1: обучаем только декодер (быстро)
freeze_encoder(model_smp)
optimizer_smp = optim.Adam(
    filter(lambda p: p.requires_grad, model_smp.parameters()),
    lr=1e-3
)
criterion_smp = DiceBCELoss(weight=0.5).to(device)

print(f"Обучаемых параметров: {sum(p.numel() for p in model_smp.parameters() if p.requires_grad):,}")


In [ ]:
# ── Этап 1: обучение с замороженным энкодером ────────────────────────────────
print("=== Этап 1: Только декодер ===")
history_smp1 = fit(model_smp, train_loader, val_loader,
                   optimizer_smp, criterion_smp,
                   epochs=5, device=device, save_path='smp_stage1.pt')

# ── Этап 2: fine-tune всей сети ───────────────────────────────────────────────
print("\n=== Этап 2: Fine-tune вся сеть ===")
unfreeze_encoder(model_smp)
optimizer_smp2 = optim.Adam(model_smp.parameters(), lr=1e-4)  # меньший LR!
scheduler_smp  = optim.lr_scheduler.CosineAnnealingLR(optimizer_smp2, T_max=5)

history_smp2 = fit(model_smp, train_loader, val_loader,
                   optimizer_smp2, criterion_smp,
                   epochs=5, device=device,
                   scheduler=scheduler_smp, save_path='smp_best.pt')


In [ ]:
# ── FPN — Feature Pyramid Network ─────────────────────────────────────────────

model_fpn = smp.FPN(
    encoder_name    = "resnet34",
    encoder_weights = "imagenet",
    in_channels     = 3,
    classes         = 1,
).to(device)

print("FPN параметров:", f"{sum(p.numel() for p in model_fpn.parameters()):,}")

optimizer_fpn = optim.Adam(model_fpn.parameters(), lr=1e-3)

print("\n=== Обучаем FPN ===")
history_fpn = fit(model_fpn, train_loader, val_loader,
                  optimizer_fpn, criterion_smp,
                  epochs=5, device=device, save_path='fpn_best.pt')


---
## 📊 Сравнение моделей

Сравниваем три модели:
- `UNet` (обученный с нуля)
- `SMP UNet` (ResNet34 backbone, pretrained ImageNet)
- `SMP FPN` (ResNet34 backbone, pretrained ImageNet)


In [ ]:
@torch.no_grad()
def evaluate_model(model, loader, device, threshold=0.5):
    """Считает Dice и IoU по всему датасету."""
    model.eval()
    dice_sum = iou_sum = 0
    n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = torch.sigmoid(model(x)).squeeze(1)
        for i in range(pred.shape[0]):
            p = (pred[i] > threshold).float().cpu()
            g = y[i].float().cpu()
            dice_sum += dice_coeff(p, g).item()
            iou_sum  += iou_score(p, g).item()
            n += 1
    return dice_sum / n, iou_sum / n


models_to_compare = {
    "UNet (scratch)":     ('unet_best.pt',  UNet(n_channels=3, n_classes=1, base_ch=32)),
    "SMP UNet (ResNet34)":('smp_best.pt',   smp.Unet("resnet34", encoder_weights=None, in_channels=3, classes=1)),
    "SMP FPN (ResNet34)": ('fpn_best.pt',   smp.FPN("resnet34",  encoder_weights=None, in_channels=3, classes=1)),
}

results = {}
for name, (path, m) in models_to_compare.items():
    try:
        m.load_state_dict(torch.load(path, map_location=device))
        m = m.to(device)
        dice, iou = evaluate_model(m, val_loader, device)
        results[name] = {'Dice': dice, 'IoU': iou}
        print(f"{name:30s}  Dice={dice*100:.1f}%  IoU={iou*100:.1f}%")
    except Exception as e:
        print(f"{name}: не удалось загрузить ({e})")


In [ ]:
# ── Визуальное сравнение ──────────────────────────────────────────────────────
if results:
    names  = list(results.keys())
    dices  = [results[n]['Dice']*100 for n in names]
    ious   = [results[n]['IoU']*100  for n in names]

    x = np.arange(len(names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))
    bars1 = ax.bar(x - width/2, dices, width, label='Dice (%)', color='steelblue', alpha=0.8)
    bars2 = ax.bar(x + width/2, ious,  width, label='IoU (%)',  color='coral',     alpha=0.8)

    for bar in bars1 + bars2:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9)

    ax.set_xticks(x); ax.set_xticklabels(names, rotation=10, ha='right')
    ax.set_ylabel('Score (%)'); ax.set_title('Сравнение моделей сегментации')
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, 100)
    plt.tight_layout(); plt.show()


In [ ]:
# ── Визуальное сравнение предсказаний ─────────────────────────────────────────
def compare_predictions(models_dict, dataset, n=3, threshold=0.5, device=device):
    """Показывает предсказания нескольких моделей рядом."""
    model_names = list(models_dict.keys())
    n_cols = 2 + len(model_names)   # img + gt + каждая модель

    fig, axes = plt.subplots(n, n_cols, figsize=(3*n_cols, 3*n))

    for row in range(n):
        img, mask_gt = dataset[row]

        axes[row, 0].imshow(denormalize(img)); axes[row, 0].axis('off')
        axes[row, 1].imshow(mask_gt.numpy(), cmap='gray'); axes[row, 1].axis('off')

        for col, (name, m) in enumerate(models_dict.items(), start=2):
            m.eval()
            with torch.no_grad():
                pred = torch.sigmoid(m(img.unsqueeze(0).to(device)))
                pred = (pred.squeeze().cpu().numpy() >= threshold).astype(float)
            axes[row, col].imshow(pred, cmap='gray')
            axes[row, col].axis('off')

    # Заголовки
    for j, title in enumerate(['Изображение', 'Ground Truth'] + model_names):
        axes[0, j].set_title(title, fontsize=9)

    plt.suptitle('Сравнение предсказаний', fontsize=13)
    plt.tight_layout(); plt.show()


# Загружаем все модели
loaded_models = {}
unet_scratch = UNet(n_channels=3, n_classes=1, base_ch=32)
unet_scratch.load_state_dict(torch.load('unet_best.pt', map_location=device))
loaded_models['UNet (scratch)'] = unet_scratch.to(device)

try:
    smp_unet = smp.Unet("resnet34", encoder_weights=None, in_channels=3, classes=1)
    smp_unet.load_state_dict(torch.load('smp_best.pt', map_location=device))
    loaded_models['SMP UNet'] = smp_unet.to(device)
except: pass

try:
    smp_fpn = smp.FPN("resnet34", encoder_weights=None, in_channels=3, classes=1)
    smp_fpn.load_state_dict(torch.load('fpn_best.pt', map_location=device))
    loaded_models['SMP FPN'] = smp_fpn.to(device)
except: pass

compare_predictions(loaded_models, val_dataset, n=3)


---
## 📝 Задания

### Задание 1 — Модифицируйте UNet

Реализуйте `UNet_Lite` — облегчённую версию с меньшим числом параметров:
- Только 3 уровня энкодера (вместо 4)
- `base_ch = 16`
- Без Bottleneck

Обучите и сравните с полным UNet. Насколько меньше параметров?

```python
class UNet_Lite(nn.Module):
    # TODO: реализуйте облегчённую версию
    pass
```


In [ ]:
class UNet_Lite(nn.Module):
    """Задание 1: Облегчённый UNet — 3 уровня, base_ch=16."""
    def __init__(self, n_channels=3, n_classes=1, base_ch=16):
        super().__init__()
        # TODO: добавьте enc1, enc2, enc3, dec3, dec2, dec1, out_conv
        # Используйте DoubleConv и EncoderBlock из кода выше
        pass

    def forward(self, x):
        # TODO: пропишите forward pass
        pass


# ── Проверка (раскомментируйте после реализации) ──────────────────────────────
# model_lite = UNet_Lite()
# with torch.no_grad():
#     y = model_lite(torch.randn(1, 3, 256, 256))
# assert y.shape == (1, 1, 256, 256), f"Неверная форма: {y.shape}"
# total = sum(p.numel() for p in model_lite.parameters())
# full  = sum(p.numel() for p in UNet(base_ch=32).parameters())
# print(f"UNet_Lite: {total:,} параметров")
# print(f"UNet full: {full:,} параметров  (в {full/total:.1f}x больше)")


### Задание 2 — Попробуйте DeepLabV3+

Используйте `smp.DeepLabV3Plus` с другим энкодером.
Сравните с UNet и FPN по метрикам.

```python
model_dlv3 = smp.DeepLabV3Plus(
    encoder_name    = "???",   # выберите: resnet50, efficientnet-b1, mobilenet_v2
    encoder_weights = "imagenet",
    in_channels     = 3,
    classes         = 1,
)
```

Вопросы для анализа:
1. Какая архитектура лучше по Dice?
2. Какая обучается быстрее?
3. Как влияет выбор backbone?


In [ ]:
# TODO: обучите DeepLabV3Plus и сравните с другими моделями

# model_dlv3 = smp.DeepLabV3Plus(
#     encoder_name    = "...",
#     encoder_weights = "imagenet",
#     in_channels     = 3,
#     classes         = 1,
# ).to(device)

# history_dlv3 = fit(model_dlv3, train_loader, val_loader,
#                    optimizer, criterion_smp,
#                    epochs=10, device=device, save_path='dlv3_best.pt')


### Задание 3 — Эксперимент с Loss функцией

Сравните три варианта функции потерь для UNet:
1. `DiceBCELoss(weight=0.5)` — текущий вариант
2. `DiceBCELoss(weight=0.0)` — только Dice
3. `smp.losses.FocalLoss` — фокус на трудных примерах

```python
from smp.losses import FocalLoss
criterion_focal = FocalLoss(mode='binary').to(device)
```

Постройте графики Dice и IoU для каждого варианта на одном plot.


In [ ]:
# TODO: сравните три функции потерь

# loss_variants = {
#     "BCE+Dice (0.5)": DiceBCELoss(weight=0.5),
#     "Dice only":       DiceBCELoss(weight=0.0),
#     "Focal":          smp.losses.FocalLoss(mode='binary'),
# }

# results_loss = {}
# for name, crit in loss_variants.items():
#     torch_seed(42)
#     m = UNet(n_channels=3, n_classes=1, base_ch=32).to(device)
#     opt = optim.Adam(m.parameters(), lr=1e-3)
#     h = fit(m, train_loader, val_loader, opt, crit.to(device),
#             epochs=5, device=device, save_path=f'loss_{name}.pt')
#     results_loss[name] = h
#     print(f"{name}: val Dice = {h['val_dice'][-1]*100:.1f}%")


---
## 📋 Итоги воркшопа

| Тема | Ключевые идеи |
|:----:|:-------------|
| **UNet** | Skip-connections сохраняют детали; Bottleneck — семантика |
| **Метрики** | Dice ≈ F1, IoU = Jaccard; порог бинаризации влияет на результат |
| **Augmentations** | Albumentations применяет одинаковые трансформации к img и mask |
| **SMP** | Pretrained backbone → Transfer Learning → быстрый старт |
| **FPN** | Мультимасштабные признаки → лучше для объектов разных размеров |
| **Transfer Learning** | Freeze → train decoder → unfreeze → fine-tune |

### Полезные ссылки
- [SMP документация](https://smp.readthedocs.io/)
- [Albumentations сегментация](https://albumentations.ai/docs/examples/pytorch_semantic_segmentation/)
- [Loss functions library](https://www.kaggle.com/code/bigironsphere/loss-function-library-keras-pytorch/)
- [Papers With Code: Segmentation](https://paperswithcode.com/task/semantic-segmentation)
